# Getting the final results 

In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. LOAD ALL YEARLY CSVs
# ============================================================
#ATTENTION HERE YOU NEED TO CHANGE DEPENDING ON THE TEST SUITE YOU ARE WORKING WITH
folder = Path("results")
all_files = sorted(folder.glob("best_algos_*.csv"))

dfs = []
for f in all_files:
    year = int(f.stem.split("_")[-1])
    df = pd.read_csv(f)
    df["year"] = year
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# ============================================================
# 2. COMPUTE GLOBAL BEST ALGORITHM FOR EACH (dim, f, target)
# ============================================================
#HERE REVIEW WHAT THIS PART ACTUALLY DOES:  
# DOES 

idx = df_all.groupby(["dimension", "function_id", "target"])["best_ERT"].idxmin()

df_best_overall = df_all.loc[
    idx, ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]
].sort_values(["dimension", "function_id", "target"]).reset_index(drop=True)

print("Loaded global best algorithms.")
print(df_best_overall.head())

# ============================================================
# 3. BUILD A RANKING TABLE (FOR A GIVEN TARGET PRECISION)
# ============================================================

def build_algo_ranking_table(df, target):
    """
    Build table where each row = one (dimension,function), sorted by increasing best ERT.
    Each row contains the ordering of algorithms by performance.
    """
    df_t = df[df["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target precision = {target}")

    # For each (dim, func), sort algorithms by ERT
    rows = []
    index = []

    for (dim, func), group in df_t.groupby(["dimension", "function_id"]):
        g = group.sort_values("best_ERT")
        algos = g["best_algorithm"].values
        rows.append(algos)
        index.append((dim, func))

    # Pad variable row lengths with NaN so it fits into a DataFrame
    max_len = max(len(row) for row in rows)
    padded = [list(row) + [None]*(max_len - len(row)) for row in rows]

    table = pd.DataFrame(padded, index=pd.MultiIndex.from_tuples(index, names=["dim","func"]))
    return table


# ============================================================
# 4. GET TOP ALGORITHMS UNTIL AT LEAST N_min UNIQUE ARE FOUND
# ============================================================

def get_top_algorithms(algo_ranking_table, N_min):
    seen_algos = set()
    row_idx = 0

    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()
        seen_algos.update(row_algos)
        row_idx += 1

    algo_list = sorted(seen_algos)

    print(f"\nRequested: {N_min} algorithms")
    print(f"Returning: {len(algo_list)} unique algorithms (reached rank row {row_idx}).")
    print("\nSelected algorithms:")
    for algo in algo_list:
        print(" -", algo)

    return algo_list


# ============================================================
# 5. INTERACTIVE USER INPUT
# ============================================================

try:
    print("\nAvailable targets:")
    print(df_best_overall["target"].unique())

    target_input = float(input("\nSelect a target precision (example: 1e-8): "))

    # Build ranking table for this target
    ranking_table = build_algo_ranking_table(df_best_overall, target_input)
    print(f"\nRanking table built for target = {target_input}")

    # Ask number of algorithms
    N_input = int(input("How many best algorithms do you want? (minimum = 6): "))
    if N_input < 6:
        print("Minimum number is 6 → using N = 6")
        N_input = 6

    # Compute the final list
    selected_algos = get_top_algorithms(ranking_table, N_input)

except ValueError:
    print("Invalid input. Please enter correct numbers for target and N.")


Loaded global best algorithms.
   dimension  function_id  year   best_algorithm        target  best_ERT
0          2            1  2010  HCMA_loshchilov  1.000000e-08  6.000000
1          2            1  2010  HCMA_loshchilov  1.000000e-05  6.000000
2          2            1  2010  HCMA_loshchilov  1.000000e-03  6.000000
3          2            1  2010  HCMA_loshchilov  1.000000e-02  6.000000
4          2            1  2009       NEWUOA_ros  1.000000e-01  5.666667

Available targets:
[1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (example: 1e-8): 1e-8

Ranking table built for target = 1e-08
How many best algorithms do you want? (minimum = 6): 7

Requested: 7 algorithms
Returning: 7 unique algorithms (reached rank row 7).

Selected algorithms:
 - HCMA_loshchilov
 - LSfminbnd_posik
 - LSstep_posik
 - NELDERDOERR_doerr
 - Powell-scipy-2019_Varelas
 - SHADE-LM-POP4-to-10_Okulewicz
 - lmm-CMA-ES_auger


In [2]:
def make_dimension_ranking_table(df_best_overall, target):
    """
    Build the dimension-wise ranking table:
    Rows = rank (1st best, 2nd best...)
    Columns = dimension (2, 3, 5, 10, 20, 40, ...)
    Values = algorithm name

    Counts how many times each algorithm was best for (fct_id, target)
    and ranks algorithms separately inside each dimension.
    """
    # Filter for chosen target
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    
    if df_t.empty:
        raise ValueError(f"No data found for target precision = {target}")

    # === 4. Aggregate: how many times each algorithm was best per dimension ===
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    algo_counts = algo_counts.sort_values(
        ["dimension", "count"],
        ascending=[True, False]
    )

    # Add ranking inside each dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # === 5. Pivot: rank × dimension ===
    ranking_table = algo_counts.pivot(
        index="rank",
        columns="dimension",
        values="best_algorithm"
    )

    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns),
        axis=1
    )

    print(f"\n📌 Ranking table for target precision = {target}\n")
    from IPython.display import display
    display(ranking_table)

    return ranking_table
try:
    print("\nAvailable targets:")
    print(df_best_overall["target"].unique())

    target_choice = float(input("\nChoose a target precision (e.g., 1e-8): "))

    algo_ranking_table = make_dimension_ranking_table(df_best_overall, target_choice)

except ValueError:
    print("Invalid input. Please enter a numeric target (e.g., 1e-8).")



Available targets:
[1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Choose a target precision (e.g., 1e-8): 1e-8

📌 Ranking table for target precision = 1e-08



dimension,2,3,5,10,20,40
rank,,,,,,
1,DE-BFGS_voglis,lq-CMA-ES_Hansen,HE-ES_Glasmachers,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov
2,DTS-CMA-ES_Pitra,GLOBAL_pal,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMAES-APOP-KMA_Nguyen,BIPOP-aCMA-STEP_loshchilov
3,HMLSL_pal,NELDER_hansen,GLOBAL_pal,BIPOPsaACM_loshchilov,NIPOPaCMA_loshchilov,CMAES-APOP-KMA_Nguyen
4,NELDERDOERR_doerr,BIPOP-aCMA-STEP_loshchilov,PSA-CMA-ES_Nishida,FULLNEWUOA_ros,BIPOP-aCMA-STEP_loshchilov,NEWUOA_ros
5,SHADE-LM-POP4-to-10_Okulewicz,DASA_korosec,lq-CMA-ES_Hansen,PSA-CMA-ES_Nishida,CMA-ES-Akimoto_Gharafi,NIPOPaCMA_loshchilov
6,SHADE-LM_Okulewicz,DE-BFGS_voglis,CMAES-APOP-Var1_Nguyen,SLSQP+lq-CMA-ES_Hansen,COBYLA-scipy-2019_Varelas,BFGS-P-09_Blelly
7,fmincon_pal,DE-scipy-2019_Varelas,DEctpb_posik,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMA-ES-Akimoto_Gharafi
8,BFGS-P-StPt_Blelly,DTS-CMA-ES_Pitra,DTS-CMA-ES_Pitra,BIRMIN_Kudela,HE-ES_Glasmachers,CMAES_Hutter_hutter
9,DIRECT-REV_Kudela,HE-ES_Glasmachers,HMLSL_pal,CMAES-APOP-MA_Nguyen,IPOPsaACM_loshchilov,LSfminbnd_posik


In [3]:
import pandas as pd
from pathlib import Path

# --- assuming df_best_overall is already built as before ---
# columns: ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]

def make_dimension_ranking_table(df_best_overall, target):
    """
    For a given target:
    - Count, for each dimension, how many times each algorithm was best.
    - Rank algorithms within each dimension by this count.
    - Pivot to get a table: rows = rank, columns = dimension, values = algorithm name.
    """
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    # Aggregate: how many times each algorithm is best per dimension
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    # Sort within each dimension by count
    algo_counts = algo_counts.sort_values(
        ["dimension", "count"], ascending=[True, False]
    )

    # Add rank per dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Pivot: rows = rank, columns = dimension
    ranking_table = algo_counts.pivot(
        index="rank", columns="dimension", values="best_algorithm"
    )

    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns),
        axis=1
    )

    return ranking_table


In [4]:
def get_top_algorithms_from_rank_table(rank_table, N_min):
    """
    Take the dimension×rank table:
    - Start from rank 1, then rank 2, etc.
    - Collect all algorithms that appear in each row (all dimensions).
    - Stop when we have at least N_min unique algorithms.
    """
    seen = set()
    row_idx = 0

    while len(seen) < N_min and row_idx < len(rank_table):
        row_algos = rank_table.iloc[row_idx].dropna().unique()
        seen.update(row_algos)
        row_idx += 1

    algo_list = sorted(seen)

    print(f"\nRequested: {N_min} algorithms")
    print(f"Returning: {len(algo_list)} unique algorithms (used ranks 1–{row_idx}).\n")
    print("Selected algorithms:")
    for a in algo_list:
        print(" -", a)

    return algo_list


In [7]:
def get_top_algorithms_from_rank_table(rank_table, N_min):
    """
    From the dimension×rank table:
    - Iterate rank rows in order (1,2,3,...).
    - Collect algorithms left → right (dimension order).
    - Preserve order of first appearance.
    - Stop when at least N_min unique algorithms are collected.
    """

    ordered_algos = []     # keeps order of appearance
    seen = set()           # for fast membership check

    row_idx = 0

    while len(seen) < N_min and row_idx < len(rank_table):
        row = rank_table.iloc[row_idx]

        # Loop across dimensions in display order
        for algo in row.dropna():
            if algo not in seen:
                seen.add(algo)
                ordered_algos.append(algo)

        row_idx += 1

    print(f"\nRequested: {N_min} algorithms")
    print(f"Returning: {len(ordered_algos)} algorithms (used ranks 1–{row_idx}).\n")

    print("Selected algorithms (ordered by ranking row):")
    for algo in ordered_algos:
        print(" -", algo)

    return ordered_algos


In [20]:
try:
    print("Available targets:", df_best_overall["target"].unique())

    target_input = float(input("\nSelect a target precision (e.g., 1e-8 or 1e-5): "))

    algo_ranking_table = make_dimension_ranking_table(df_best_overall, target_input)
    print(f"\nRanking table built for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table.head(10))

    N_input = int(input("\nHow many best algorithms do you want? (minimum = 6): "))
    if N_input < 6:
        print("Minimum number of best algorithms is 6. Using N = 6.")
        N_input = 6

    best_algos = get_top_algorithms_from_rank_table(algo_ranking_table, N_input)

except ValueError:
    print("Invalid input: please enter numeric values for target and N.")


Available targets: [1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (e.g., 1e-8 or 1e-5): 1e-8

Ranking table built for target = 1e-08


dimension,2,3,5,10,20,40
rank,,,,,,
1,DE-BFGS_voglis,lq-CMA-ES_Hansen,HE-ES_Glasmachers,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov
2,DTS-CMA-ES_Pitra,GLOBAL_pal,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMAES-APOP-KMA_Nguyen,BIPOP-aCMA-STEP_loshchilov
3,HMLSL_pal,NELDER_hansen,GLOBAL_pal,BIPOPsaACM_loshchilov,NIPOPaCMA_loshchilov,CMAES-APOP-KMA_Nguyen
4,NELDERDOERR_doerr,BIPOP-aCMA-STEP_loshchilov,PSA-CMA-ES_Nishida,FULLNEWUOA_ros,BIPOP-aCMA-STEP_loshchilov,NEWUOA_ros
5,SHADE-LM-POP4-to-10_Okulewicz,DASA_korosec,lq-CMA-ES_Hansen,PSA-CMA-ES_Nishida,CMA-ES-Akimoto_Gharafi,NIPOPaCMA_loshchilov
6,SHADE-LM_Okulewicz,DE-BFGS_voglis,CMAES-APOP-Var1_Nguyen,SLSQP+lq-CMA-ES_Hansen,COBYLA-scipy-2019_Varelas,BFGS-P-09_Blelly
7,fmincon_pal,DE-scipy-2019_Varelas,DEctpb_posik,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMA-ES-Akimoto_Gharafi
8,BFGS-P-StPt_Blelly,DTS-CMA-ES_Pitra,DTS-CMA-ES_Pitra,BIRMIN_Kudela,HE-ES_Glasmachers,CMAES_Hutter_hutter
9,DIRECT-REV_Kudela,HE-ES_Glasmachers,HMLSL_pal,CMAES-APOP-MA_Nguyen,IPOPsaACM_loshchilov,LSfminbnd_posik



How many best algorithms do you want? (minimum = 6): 7

Requested: 7 algorithms
Returning: 9 algorithms (used ranks 1–2).

Selected algorithms (ordered by ranking row):
 - DE-BFGS_voglis
 - lq-CMA-ES_Hansen
 - HE-ES_Glasmachers
 - BIPOP-saACM-k_loshchilov
 - DTS-CMA-ES_Pitra
 - GLOBAL_pal
 - BIPOP-aCMA-STEP_loshchilov
 - HCMA_loshchilov
 - CMAES-APOP-KMA_Nguyen


In [21]:
cocopp.main(['DE-BFGS_voglis'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2012\DE-BFGS_voglis_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\DE-BFGS_voglis_noiseless_111314h3854
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 14:39:05 2025).
  done (Thu Nov 13 14:39:36 2025).
Generating LaTeX tables...
  done (Thu Nov 13 14:39:37 2025).
ECDF graphs...
  done (Thu Nov 13 14:39:48 2025).
ECDF graphs per function...
  done (Thu Nov 13 14:40:45 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 14:40:50 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\DE-BFGS_voglis_noiseless_111314h3854
Setting changes in `cocopp.generi

DictAlg([(('DE-BFGS_voglis', ''),
          [DataSet(DE-BFGS_voglis on f1 2-D),
           DataSet(DE-BFGS_voglis on f2 2-D),
           DataSet(DE-BFGS_voglis on f3 2-D),
           DataSet(DE-BFGS_voglis on f4 2-D),
           DataSet(DE-BFGS_voglis on f5 2-D),
           DataSet(DE-BFGS_voglis on f6 2-D),
           DataSet(DE-BFGS_voglis on f7 2-D),
           DataSet(DE-BFGS_voglis on f8 2-D),
           DataSet(DE-BFGS_voglis on f9 2-D),
           DataSet(DE-BFGS_voglis on f10 2-D),
           DataSet(DE-BFGS_voglis on f11 2-D),
           DataSet(DE-BFGS_voglis on f12 2-D),
           DataSet(DE-BFGS_voglis on f13 2-D),
           DataSet(DE-BFGS_voglis on f14 2-D),
           DataSet(DE-BFGS_voglis on f15 2-D),
           DataSet(DE-BFGS_voglis on f16 2-D),
           DataSet(DE-BFGS_voglis on f17 2-D),
           DataSet(DE-BFGS_voglis on f18 2-D),
           DataSet(DE-BFGS_voglis on f19 2-D),
           DataSet(DE-BFGS_voglis on f20 2-D),
           DataSet(DE-BFGS_voglis o

In [26]:
cocopp.main(['DE-BFGS_voglis','bbob/2020/lq-CMA-ES_Hansen.tgz'])
            


Post-processing (2+)
  Using 2 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2012\DE-BFGS_voglis_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2012\DE-BFGS_voglis_noiseless.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
  Will generate output data in folder ppdata\DE-BF_lq-CM_111316h3548
    this might take several minutes.
ECDF runlength ratio graphs...
  done (Thu Nov 13 16:36:07 2025).
ECDF runlength graphs...
  done (Thu Nov 13 16:36:34 2025).
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 16:36:35 2025).
  done (Thu Nov 13 16:36:53 2025).
ECDF graphs per function group...
  done 

DictAlg([(('DE-BFGS_voglis', ''),
          [DataSet(DE-BFGS_voglis on f1 2-D),
           DataSet(DE-BFGS_voglis on f2 2-D),
           DataSet(DE-BFGS_voglis on f3 2-D),
           DataSet(DE-BFGS_voglis on f4 2-D),
           DataSet(DE-BFGS_voglis on f5 2-D),
           DataSet(DE-BFGS_voglis on f6 2-D),
           DataSet(DE-BFGS_voglis on f7 2-D),
           DataSet(DE-BFGS_voglis on f8 2-D),
           DataSet(DE-BFGS_voglis on f9 2-D),
           DataSet(DE-BFGS_voglis on f10 2-D),
           DataSet(DE-BFGS_voglis on f11 2-D),
           DataSet(DE-BFGS_voglis on f12 2-D),
           DataSet(DE-BFGS_voglis on f13 2-D),
           DataSet(DE-BFGS_voglis on f14 2-D),
           DataSet(DE-BFGS_voglis on f15 2-D),
           DataSet(DE-BFGS_voglis on f16 2-D),
           DataSet(DE-BFGS_voglis on f17 2-D),
           DataSet(DE-BFGS_voglis on f18 2-D),
           DataSet(DE-BFGS_voglis on f19 2-D),
           DataSet(DE-BFGS_voglis on f20 2-D),
           DataSet(DE-BFGS_voglis o

In [27]:
cocopp.main(['HE-ES_Glasmachers'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\HE-ES_Glasmachers_111317h5925
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:01:02 2025).
  done (Thu Nov 13 18:02:12 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:02:14 2025).
ECDF graphs...
  done (Thu Nov 13 18:02:35 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:04:36 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:04:49 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\HE-ES_Glasmachers_111317h5925
Setting changes in `cocopp.genericsettings` compared t

DictAlg([(('HE-ES_Glasmachers', ''),
          [DataSet(HE-ES_Glasmachers on f1 2-D),
           DataSet(HE-ES_Glasmachers on f2 2-D),
           DataSet(HE-ES_Glasmachers on f3 2-D),
           DataSet(HE-ES_Glasmachers on f4 2-D),
           DataSet(HE-ES_Glasmachers on f5 2-D),
           DataSet(HE-ES_Glasmachers on f6 2-D),
           DataSet(HE-ES_Glasmachers on f7 2-D),
           DataSet(HE-ES_Glasmachers on f8 2-D),
           DataSet(HE-ES_Glasmachers on f9 2-D),
           DataSet(HE-ES_Glasmachers on f10 2-D),
           DataSet(HE-ES_Glasmachers on f11 2-D),
           DataSet(HE-ES_Glasmachers on f12 2-D),
           DataSet(HE-ES_Glasmachers on f13 2-D),
           DataSet(HE-ES_Glasmachers on f14 2-D),
           DataSet(HE-ES_Glasmachers on f15 2-D),
           DataSet(HE-ES_Glasmachers on f16 2-D),
           DataSet(HE-ES_Glasmachers on f17 2-D),
           DataSet(HE-ES_Glasmachers on f18 2-D),
           DataSet(HE-ES_Glasmachers on f19 2-D),
           DataSet(HE-

In [28]:
cocopp.main(['BIPOP-saACM-k_loshchilov'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\BIPOP-saACM-k_loshchilov_noiseless_111318h0450
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:05:05 2025).
  done (Thu Nov 13 18:06:24 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:06:26 2025).
ECDF graphs...
  done (Thu Nov 13 18:06:45 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:09:03 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:09:15 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\BIPOP-saACM-k_loshchilov_noiseless_111318h0450
Set

DictAlg([(('BIPOP-saACM-k_loshchilov', ''),
          [DataSet(BIPOP-saACM-k_loshchilov on f1 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f2 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f3 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f4 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f5 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f6 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f7 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f8 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f9 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f10 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f11 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f12 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f13 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f14 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f15 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f16 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f17 2-

In [29]:
cocopp.main([' bbob/2017/DTS-CMA-ES_Pitra.tgz'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2017\DTS-CMA-ES_Pitra.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\DTS-CMA-ES_Pitra_111318h0915
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:09:33 2025).
  done (Thu Nov 13 18:10:51 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:10:53 2025).
ECDF graphs...
  done (Thu Nov 13 18:11:14 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:13:20 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:13:33 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\DTS-CMA-ES_Pitra_111318h0915
Setting changes in `cocopp.genericsettings` compared to d

DictAlg([(('DTS-CMA-ES_Pitra', ''),
          [DataSet(DTS-CMA-ES_Pitra on f1 2-D),
           DataSet(DTS-CMA-ES_Pitra on f2 2-D),
           DataSet(DTS-CMA-ES_Pitra on f3 2-D),
           DataSet(DTS-CMA-ES_Pitra on f4 2-D),
           DataSet(DTS-CMA-ES_Pitra on f5 2-D),
           DataSet(DTS-CMA-ES_Pitra on f6 2-D),
           DataSet(DTS-CMA-ES_Pitra on f7 2-D),
           DataSet(DTS-CMA-ES_Pitra on f8 2-D),
           DataSet(DTS-CMA-ES_Pitra on f9 2-D),
           DataSet(DTS-CMA-ES_Pitra on f10 2-D),
           DataSet(DTS-CMA-ES_Pitra on f11 2-D),
           DataSet(DTS-CMA-ES_Pitra on f12 2-D),
           DataSet(DTS-CMA-ES_Pitra on f13 2-D),
           DataSet(DTS-CMA-ES_Pitra on f14 2-D),
           DataSet(DTS-CMA-ES_Pitra on f15 2-D),
           DataSet(DTS-CMA-ES_Pitra on f16 2-D),
           DataSet(DTS-CMA-ES_Pitra on f17 2-D),
           DataSet(DTS-CMA-ES_Pitra on f18 2-D),
           DataSet(DTS-CMA-ES_Pitra on f19 2-D),
           DataSet(DTS-CMA-ES_Pitra on f20

In [30]:
cocopp.main(['bbob/2009/GLOBAL_pal_noiseless.tgz'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2009\GLOBAL_pal_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\GLOBAL_pal_noiseless_111318h1334
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:13:45 2025).
  done (Thu Nov 13 18:15:02 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:15:04 2025).
ECDF graphs...
  done (Thu Nov 13 18:15:29 2025).
ECDF graphs per function...
Crafting effort for ('GLOBAL_pal', '') is 0.5117
Crafting effort for ('GLOBAL_pal', '') is 0.5117
Crafting effort for ('GLOBAL_pal', '') is 0.5117
Crafting effort for ('GLOBAL_pal', '') is 0.5117
Crafting effort for ('GLOBAL_pal', '') is 0.5117
Crafting effort

DictAlg([(('GLOBAL_pal', ''),
          [DataSet(GLOBAL_pal on f1 2-D),
           DataSet(GLOBAL_pal on f2 2-D),
           DataSet(GLOBAL_pal on f3 2-D),
           DataSet(GLOBAL_pal on f4 2-D),
           DataSet(GLOBAL_pal on f5 2-D),
           DataSet(GLOBAL_pal on f6 2-D),
           DataSet(GLOBAL_pal on f7 2-D),
           DataSet(GLOBAL_pal on f8 2-D),
           DataSet(GLOBAL_pal on f9 2-D),
           DataSet(GLOBAL_pal on f10 2-D),
           DataSet(GLOBAL_pal on f11 2-D),
           DataSet(GLOBAL_pal on f12 2-D),
           DataSet(GLOBAL_pal on f13 2-D),
           DataSet(GLOBAL_pal on f14 2-D),
           DataSet(GLOBAL_pal on f15 2-D),
           DataSet(GLOBAL_pal on f16 2-D),
           DataSet(GLOBAL_pal on f17 2-D),
           DataSet(GLOBAL_pal on f18 2-D),
           DataSet(GLOBAL_pal on f19 2-D),
           DataSet(GLOBAL_pal on f20 2-D),
           DataSet(GLOBAL_pal on f21 2-D),
           DataSet(GLOBAL_pal on f22 2-D),
           DataSet(GLOBAL_pal on 

In [32]:
cocopp.main(['BIPOP-aCMA-STEP_loshchilov'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-aCMA-STEP_loshchilov_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\BIPOP-aCMA-STEP_loshchilov_noiseless_111318h2757
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:28:21 2025).
  done (Thu Nov 13 18:29:47 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:29:50 2025).
ECDF graphs...
  done (Thu Nov 13 18:30:07 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:32:24 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:32:36 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\BIPOP-aCMA-STEP_loshchilov_noiseless_111318h27

DictAlg([(('BIPOP-aCMA-STEP_loshchilov', ''),
          [DataSet(BIPOP-aCMA-STEP_loshchilov on f1 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f2 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f3 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f4 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f5 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f6 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f7 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f8 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f9 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f10 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f11 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f12 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f13 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f14 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f15 2-D),
           DataSet(BIPOP-aCMA-STEP_loshchilov on f16 2-D),
           DataSet(

In [33]:
cocopp.main(['HCMA_loshchilov'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\HCMA_loshchilov_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\HCMA_loshchilov_noiseless_111318h3237
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:32:50 2025).
  done (Thu Nov 13 18:34:09 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:34:11 2025).
ECDF graphs...
  done (Thu Nov 13 18:34:29 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:36:36 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:36:48 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\HCMA_loshchilov_noiseless_111318h3237
Setting changes in `cocopp.gen

DictAlg([(('HCMA_loshchilov', ''),
          [DataSet(HCMA_loshchilov on f1 2-D),
           DataSet(HCMA_loshchilov on f2 2-D),
           DataSet(HCMA_loshchilov on f3 2-D),
           DataSet(HCMA_loshchilov on f4 2-D),
           DataSet(HCMA_loshchilov on f5 2-D),
           DataSet(HCMA_loshchilov on f6 2-D),
           DataSet(HCMA_loshchilov on f7 2-D),
           DataSet(HCMA_loshchilov on f8 2-D),
           DataSet(HCMA_loshchilov on f9 2-D),
           DataSet(HCMA_loshchilov on f10 2-D),
           DataSet(HCMA_loshchilov on f11 2-D),
           DataSet(HCMA_loshchilov on f12 2-D),
           DataSet(HCMA_loshchilov on f13 2-D),
           DataSet(HCMA_loshchilov on f14 2-D),
           DataSet(HCMA_loshchilov on f15 2-D),
           DataSet(HCMA_loshchilov on f16 2-D),
           DataSet(HCMA_loshchilov on f17 2-D),
           DataSet(HCMA_loshchilov on f18 2-D),
           DataSet(HCMA_loshchilov on f19 2-D),
           DataSet(HCMA_loshchilov on f20 2-D),
           Dat

In [34]:
cocopp.main(['CMAES-APOP-KMA_Nguyen'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\CMAES-APOP-KMA_Nguyen_111318h3649
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 18:37:03 2025).
  done (Thu Nov 13 18:37:33 2025).
Generating LaTeX tables...
  done (Thu Nov 13 18:37:34 2025).
ECDF graphs...
  done (Thu Nov 13 18:37:43 2025).
ECDF graphs per function...
  done (Thu Nov 13 18:38:35 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 18:38:40 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\CMAES-APOP-KMA_Nguyen_111318h3649
Setting changes in `cocopp.genericsettings

DictAlg([(('CMAES-APOP-KMA_Nguyen', ''),
          [DataSet(CMAES-APOP-KMA_Nguyen on f1 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f2 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f3 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f4 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f5 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f6 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f7 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f8 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f9 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f10 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f11 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f12 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f13 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f14 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f15 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f16 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f17 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f18 2-

In [9]:
from pathlib import Path

def resolve_algo_paths(algos):
    base = Path(r"C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob")
    paths = []

    for algo in algos:
        matches = list(base.rglob(f"{algo}*.tgz"))
        if not matches:
            print(f"❌ No file found for: {algo}")
        else:
            print(f"✔ {algo} → {matches[0].name}")
            paths.append(str(matches[0]))

    return paths

selected_algos = [
    "NELDERDOERR_doerr",
    "lq-CMA-ES_Hansen",
    "HE-ES_Glasmachers",
    "BIPOP-saACM-k_loshchilov",
    "CMAES-APOP-KMA_Nguyen",
    "BIPOP-aCMA-STEP_loshchilov",
    "DIRECT-REV_Kudela",
    "fminunc_pal",
    "OQNLP_pal",
    "SLSQP+lq-CMA-ES_Hansen",
    "NIPOPaCMA_loshchilov",
    "NEWUOA_ros"
]

algo_paths = resolve_algo_paths(selected_algos)


✔ NELDERDOERR_doerr → NELDERDOERR_doerr_noiseless.tgz
✔ lq-CMA-ES_Hansen → lq-CMA-ES_Hansen.tgz
✔ HE-ES_Glasmachers → HE-ES_Glasmachers.tgz
✔ BIPOP-saACM-k_loshchilov → BIPOP-saACM-k_loshchilov_noiseless.tgz
✔ CMAES-APOP-KMA_Nguyen → CMAES-APOP-KMA_Nguyen.tgz
✔ BIPOP-aCMA-STEP_loshchilov → BIPOP-aCMA-STEP_loshchilov_noiseless.tgz
✔ DIRECT-REV_Kudela → DIRECT-REV_Kudela.tgz
✔ fminunc_pal → fminunc_pal_noiseless.tgz
✔ OQNLP_pal → OQNLP_pal_noiseless.tgz
✔ SLSQP+lq-CMA-ES_Hansen → SLSQP+lq-CMA-ES_Hansen.tgz
✔ NIPOPaCMA_loshchilov → NIPOPaCMA_loshchilov_noiseless.tgz
✔ NEWUOA_ros → NEWUOA_ros_noiseless.tgz


In [10]:
import cocopp

cocopp.main(algo_paths)


Post-processing (2+)
  Using 12 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2009\NELDERDOERR_doerr_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-aCMA-STEP_loshchilov_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\DIRECT-REV_Kudela.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\fminunc_pal_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\OQNLP_pal_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQ

Exception: There is more than a single entry associated with folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-aCMA-STEP_loshchilov_noiseless.tgz on 2-D f1.

In [11]:
from pathlib import Path
from collections import defaultdict

base = Path(r"C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob")

duplicates = defaultdict(list)

for file in base.rglob("*.tgz"):
    key = file.name   # duplicate same-name files
    duplicates[key].append(file)

for name, files in duplicates.items():
    if len(files) > 1:
        print("\n❗ DUPLICATE FOUND:", name)
        for f in files:
            print("   →", f)


In [12]:
cocopp.main(algo_paths)


Post-processing (2+)
  Using 12 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2009\NELDERDOERR_doerr_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-aCMA-STEP_loshchilov_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\DIRECT-REV_Kudela.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\fminunc_pal_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\OQNLP_pal_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQ

Exception: There is more than a single entry associated with folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-aCMA-STEP_loshchilov_noiseless.tgz on 2-D f1.

In [13]:
cocopp.main(['lq-CMA-ES_Hansen'])

Post-processing (1)


ValueError: 'lq-CMA-ES_Hansen' has multiple matches in the data archive:
   bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz
   bbob/2020/lq-CMA-ES_Hansen.tgz
Either pick a single match, or use the `get_all` or
`get_first` method, or use the ! (first) or * (all)
marker and try again.

In [15]:
cocopp.main([' bbob/2020/lq-CMA-ES_Hansen.tgz'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\lq-CMA-ES_Hansen_111313h1119
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 13:12:58 2025).
  done (Thu Nov 13 13:13:31 2025).
Generating LaTeX tables...
  done (Thu Nov 13 13:13:32 2025).
ECDF graphs...
  done (Thu Nov 13 13:13:47 2025).
ECDF graphs per function...
  done (Thu Nov 13 13:14:44 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 13:14:52 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\lq-CMA-ES_Hansen_111313h1119
Setting changes in `cocopp.genericsettings` compared to d

DictAlg([(('lq-CMA-ES_Hansen', ''),
          [DataSet(lq-CMA-ES_Hansen on f1 2-D),
           DataSet(lq-CMA-ES_Hansen on f2 2-D),
           DataSet(lq-CMA-ES_Hansen on f3 2-D),
           DataSet(lq-CMA-ES_Hansen on f4 2-D),
           DataSet(lq-CMA-ES_Hansen on f5 2-D),
           DataSet(lq-CMA-ES_Hansen on f6 2-D),
           DataSet(lq-CMA-ES_Hansen on f7 2-D),
           DataSet(lq-CMA-ES_Hansen on f8 2-D),
           DataSet(lq-CMA-ES_Hansen on f9 2-D),
           DataSet(lq-CMA-ES_Hansen on f10 2-D),
           DataSet(lq-CMA-ES_Hansen on f11 2-D),
           DataSet(lq-CMA-ES_Hansen on f12 2-D),
           DataSet(lq-CMA-ES_Hansen on f13 2-D),
           DataSet(lq-CMA-ES_Hansen on f14 2-D),
           DataSet(lq-CMA-ES_Hansen on f15 2-D),
           DataSet(lq-CMA-ES_Hansen on f16 2-D),
           DataSet(lq-CMA-ES_Hansen on f17 2-D),
           DataSet(lq-CMA-ES_Hansen on f18 2-D),
           DataSet(lq-CMA-ES_Hansen on f19 2-D),
           DataSet(lq-CMA-ES_Hansen on f20

In [17]:
cocopp.main(['HE-ES_Glasmachers'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\HE-ES_Glasmachers_111313h2401
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 13:25:27 2025).
  done (Thu Nov 13 13:25:57 2025).
Generating LaTeX tables...
  done (Thu Nov 13 13:25:58 2025).
ECDF graphs...
  done (Thu Nov 13 13:26:05 2025).
ECDF graphs per function...
  done (Thu Nov 13 13:26:55 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 13:27:01 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\HE-ES_Glasmachers_111313h2401
Setting changes in `cocopp.genericsettings` compared t

DictAlg([(('HE-ES_Glasmachers', ''),
          [DataSet(HE-ES_Glasmachers on f1 2-D),
           DataSet(HE-ES_Glasmachers on f2 2-D),
           DataSet(HE-ES_Glasmachers on f3 2-D),
           DataSet(HE-ES_Glasmachers on f4 2-D),
           DataSet(HE-ES_Glasmachers on f5 2-D),
           DataSet(HE-ES_Glasmachers on f6 2-D),
           DataSet(HE-ES_Glasmachers on f7 2-D),
           DataSet(HE-ES_Glasmachers on f8 2-D),
           DataSet(HE-ES_Glasmachers on f9 2-D),
           DataSet(HE-ES_Glasmachers on f10 2-D),
           DataSet(HE-ES_Glasmachers on f11 2-D),
           DataSet(HE-ES_Glasmachers on f12 2-D),
           DataSet(HE-ES_Glasmachers on f13 2-D),
           DataSet(HE-ES_Glasmachers on f14 2-D),
           DataSet(HE-ES_Glasmachers on f15 2-D),
           DataSet(HE-ES_Glasmachers on f16 2-D),
           DataSet(HE-ES_Glasmachers on f17 2-D),
           DataSet(HE-ES_Glasmachers on f18 2-D),
           DataSet(HE-ES_Glasmachers on f19 2-D),
           DataSet(HE-

In [18]:
cocopp.main(['BIPOP-saACM-k_loshchilov'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\BIPOP-saACM-k_loshchilov_noiseless_111313h3126
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 13:31:36 2025).
  done (Thu Nov 13 13:32:07 2025).
Generating LaTeX tables...
  done (Thu Nov 13 13:32:08 2025).
ECDF graphs...
  done (Thu Nov 13 13:32:15 2025).
ECDF graphs per function...
  done (Thu Nov 13 13:33:03 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 13:33:08 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\BIPOP-saACM-k_loshchilov_noiseless_111313h3126
Set

DictAlg([(('BIPOP-saACM-k_loshchilov', ''),
          [DataSet(BIPOP-saACM-k_loshchilov on f1 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f2 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f3 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f4 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f5 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f6 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f7 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f8 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f9 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f10 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f11 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f12 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f13 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f14 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f15 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f16 2-D),
           DataSet(BIPOP-saACM-k_loshchilov on f17 2-

In [19]:
cocopp.main(['CMAES-APOP-KMA_Nguyen'])

Post-processing (1)
  Using 1 data set:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz

Post-processing (1)
  loading data...
  Data consistent according to consistency_check() in pproc.DataSet
  Will generate output data in folder ppdata\CMAES-APOP-KMA_Nguyen_111313h3308
    this might take several minutes.
Scaling figures...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Thu Nov 13 13:33:21 2025).
  done (Thu Nov 13 13:33:49 2025).
Generating LaTeX tables...
  done (Thu Nov 13 13:33:50 2025).
ECDF graphs...
  done (Thu Nov 13 13:34:00 2025).
ECDF graphs per function...
  done (Thu Nov 13 13:34:48 2025).
ERT loss ratio figures and tables...
  done (Thu Nov 13 13:34:53 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\CMAES-APOP-KMA_Nguyen_111313h3308
Setting changes in `cocopp.genericsettings

DictAlg([(('CMAES-APOP-KMA_Nguyen', ''),
          [DataSet(CMAES-APOP-KMA_Nguyen on f1 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f2 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f3 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f4 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f5 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f6 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f7 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f8 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f9 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f10 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f11 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f12 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f13 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f14 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f15 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f16 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f17 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f18 2-